# 9. REPORTING DASHBOARD
## Daily Customer Churn Predictor · VivaMarket Brasil

---

**INPUT:** `../data/processed/churn_predictions_YYYYMMDD.parquet`, `../data/processed/churn_diagnostics_YYYYMMDD.csv`, `../data/processed/churn_explainability_YYYYMMDD.parquet`, and `../data/processed/retention_actions_YYYYMMDD.parquet`

**OUTPUT:** `../reports/churn_monitoring_dashboard_YYYYMMDD.html`

*A compact monitoring dashboard summarizing risk mix, diagnostics, driver patterns, and retention action readiness.*


---
## 9.1. NOTEBOOK OBJECTIVE


- **Business objective:** give stakeholders a single monitoring view for predictive risk, campaign readiness, and explainability.
- **Analytical objective:** consolidate diagnostics, driver mix, and action summaries into a lightweight HTML dashboard aligned with the canonical V2C policy.

In [1]:
import json
import logging
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', force=True)
logger = logging.getLogger('nb09_reporting_dashboard')
logger.info('NB09 started: reporting dashboard.')


2026-05-06 08:33:09,548 | INFO | NB09 started: reporting dashboard.


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

run_date_tag = datetime.now(ZoneInfo('Europe/Paris')).strftime('%Y%m%d')
prediction_df = pd.read_parquet(sorted(PROCESSED_DIR.glob('churn_predictions_*.parquet'))[-1])
explainability_df = pd.read_parquet(sorted(PROCESSED_DIR.glob('churn_explainability_*.parquet'))[-1])
actions_df = pd.read_parquet(sorted(PROCESSED_DIR.glob('retention_actions_*.parquet'))[-1])
diagnostics_df = pd.read_csv(sorted(PROCESSED_DIR.glob('churn_diagnostics_*.csv'))[-1])
dashboard_path = REPORTS_DIR / f'churn_monitoring_dashboard_{run_date_tag}.html'


In [3]:
risk_mix = (
    prediction_df.groupby('risk_tier', observed=False)
    .agg(
        rows_n=('customer_unique_id', 'size'),
        avg_probability=('churn_probability', 'mean'),
        observed_churn=('observed_target', 'mean'),
    )
    .reset_index()
)
driver_mix = (
    explainability_df.groupby(['risk_tier', 'top_driver_group'], observed=False)
    .size()
    .reset_index(name='rows_n')
)
action_mix = (
    actions_df.groupby(['risk_tier', 'recommended_offer_type', 'primary_channels'], observed=False)
    .agg(
        rows_n=('customer_unique_id', 'size'),
        send_rows=('send_action_flag', 'sum'),
        control_rows=('control_group_flag', 'sum'),
        avg_discount_pct=('recommended_discount_pct', 'mean'),
    )
    .reset_index()
)
summary_metrics = diagnostics_df[diagnostics_df['section'] == 'summary'].copy() if 'section' in diagnostics_df.columns else diagnostics_df.copy()
threshold_metrics = diagnostics_df[diagnostics_df['section'] == 'thresholds'].copy() if 'section' in diagnostics_df.columns else pd.DataFrame()

html_parts = [
    '<html><head><meta charset="utf-8"><title>Churn Monitoring Dashboard</title></head><body>',
    '<h1>CHURN MONITORING DASHBOARD</h1>',
    '<h2>Risk mix</h2>', risk_mix.to_html(index=False),
    '<h2>Model summary metrics</h2>', summary_metrics.to_html(index=False),
    '<h2>Score-threshold operating table</h2>', threshold_metrics.to_html(index=False),
    '<h2>Top driver mix</h2>', driver_mix.to_html(index=False),
    '<h2>Retention action mix</h2>', action_mix.to_html(index=False),
    '<h2>Notes</h2><ul><li>Risk tiers follow the canonical V2C percentile policy: LOW = bottom 50%, MEDIUM = next 30%, HIGH = top 20%.</li><li>High risk keeps a 15% control group for measurement discipline.</li><li>The target remains highly positive, so calibration should be interpreted cautiously even though ranking performance is strong.</li></ul>',
    '</body></html>'
]
dashboard_path.write_text('\n'.join(html_parts), encoding='utf-8')
logger.info('Dashboard saved to %s', dashboard_path)
dashboard_path

2026-05-06 08:33:09,623 | INFO | Dashboard saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/reports/churn_monitoring_dashboard_20260506.html


PosixPath('/data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/reports/churn_monitoring_dashboard_20260506.html')

---
## 9.2. NOTEBOOK CLOSURE


The reporting layer now consolidates prediction quality, churn drivers, and campaign readiness into a single lightweight dashboard aligned with the canonical V2C policy. This closes the initial end-to-end notebook flow from raw data to operational retention outputs.